# 🛡️ NetWatch — Cloud Model Training & Export Station
### Google Colab GPU-Accelerated Trainer for Multi-Engine Cyber Defense

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sihcodesprit/external_hackathon/blob/main/NetWatch_Colab.ipynb)

This notebook trains all **8 detection engines and baseline models** in Google Colab's cloud environment, benchmarks their performance, and exports a single **`netwatch_trained_models.zip`** bundle that you can download and use locally on your laptop with zero local GPU/CPU load.

## 1. ⚙️ Setup Environment & Clone Repo
Clones your GitHub repository and installs all required dependencies.

In [ ]:
import os
if not os.path.exists('/content/external_hackathon'):
    !git clone https://github.com/sihcodesprit/external_hackathon.git /content/external_hackathon
%cd /content/external_hackathon

# Pull latest code and install dependencies
!git reset --hard HEAD
!git pull origin main
!pip install -r requirements.txt
!pip install pytest scapy pycloudflared

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\n✓ Setup complete! Using compute device: {device.upper()}")

## 2. 🧠 Train All 8 Detection Engines & Models
Executes high-capacity training for:
1. **LSTM Cyber World Model** (Multi-step temporal forecasting & state prediction)
2. **Risk Head & Calibrated Predictor** (Threat likelihood estimation)
3. **Random Forest Classifier** (Tree ensemble for packet/flow features)
4. **Gradient Boosting Machine (GBM)** (Non-linear decision boundary detector)
5. **Calibrated Logistic Regression** (High-speed linear baseline)
6. **Shannon Entropy Feature Scalers & Baselines**
7. **Network Graph & Topology Analyzer**
8. **MITRE ATT&CK Stage Predictor**

In [ ]:
import sys
sys.path.insert(0, '.')
import shutil
from pathlib import Path
from netwatch.pipeline import Pipeline
from netwatch.forecasting.ensemble_scorer import EnsembleScorer
from netwatch.config import MODEL_DIR, WORLD_MODEL_PATH, SCALER_PATH

print("=" * 75)
print("🚀 Starting Master Model Training in Colab...")
print("=" * 75)

# 1. Initialize Pipeline & Ingest Multi-Stage Cyber Traces
pipe = Pipeline()
print("[*] Generating and ingesting multi-stage network traces (12 traces)...")
ingest_stats = pipe.load_data(n_traces=12, seed=42)
print(f"    -> Ingested {ingest_stats['n_packets']} packets across {ingest_stats['n_states']} states.")

# 2. Train LSTM Cyber World Model
print("\n[*] Training LSTM Cyber World Model & Risk Head...")
train_stats = pipe.train()
print(f"    -> Final Train Loss: {train_stats['training'].get('final_train_loss', 'N/A'):.4f}")
print(f"    -> Final Val Loss:   {train_stats['training'].get('final_val_loss', 'N/A'):.4f}")

# 3. Train & Persist Ensemble ML Baselines (RF, GBM, Logistic Regression)
print("\n[*] Fitting and saving Multi-Tool Ensemble ML Baselines...")
ensemble = EnsembleScorer(pipeline=pipe)
baselines_path = MODEL_DIR / "ensemble_baselines.pkl"
ensemble.save_baselines(str(baselines_path))
print(f"    -> Saved ensemble baselines to: {baselines_path}")

# 4. Evaluate Models & Benchmark Baselines
print("\n[*] Evaluating performance against classical baselines and unseen attacks...")
eval_stats = pipe.evaluate(include_baselines=True, include_unseen=True)
pipe.forecast_and_simulate(k=5)

print("\n" + "=" * 75)
print("✅ ALL 8 DETECTION MODELS SUCCESSFULLY TRAINED AND SAVED!")
print("=" * 75)

## 3. 📦 Package Models into `netwatch_trained_models.zip`
Collects all model weights, scalers, checkpoints, and registries into a single portable zip archive.

In [ ]:
import zipfile
import os
from pathlib import Path

zip_filename = "netwatch_trained_models.zip"
if os.path.exists(zip_filename):
    os.remove(zip_filename)

files_to_bundle = [
    "data/data/models/world_model_lstm.pt",
    "data/data/models/feature_scaler.pkl",
    "data/data/models/ensemble_baselines.pkl",
    "data/data/models/stage_predictor.pt",
    "models/checkpoints/registry.json",
    "data/data/generated/network_states.jsonl",
]

print(f"Creating {zip_filename}...")
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zf:
    for rel_path in files_to_bundle:
        if os.path.exists(rel_path):
            zf.write(rel_path, arcname=os.path.basename(rel_path))
            size_kb = os.path.getsize(rel_path) / 1024
            print(f"  + Packaged {rel_path} ({size_kb:.1f} KB)")
        else:
            print(f"  - Skipping missing file: {rel_path}")

total_size_mb = os.path.getsize(zip_filename) / (1024 * 1024)
print(f"\n✓ Successfully built {zip_filename} ({total_size_mb:.2f} MB)!")

## 4. ⬇️ Download Trained Model Package to Your Laptop
Click the cell below to trigger an automatic 1-click browser download of `netwatch_trained_models.zip`.

In [ ]:
from google.colab import files

print("Downloading netwatch_trained_models.zip to your computer...")
files.download('netwatch_trained_models.zip')

print("\n👉 Once downloaded, run this on your laptop terminal:")
print("   python import_models.py")

## 5. 🌐 (Optional) Launch Live SOC Web Dashboard on Cloudflare
If you want to view and test the live web app directly in the cloud before or after downloading the models:

In [ ]:
import subprocess
import time
from pycloudflared import try_cloudflare

# Start Flask app in the background
server_proc = subprocess.Popen(["python", "run.py", "--no-pipeline", "--host", "0.0.0.0", "--port", "5000"])
time.sleep(3)

# Open secure public HTTPS tunnel
tunnel_info = try_cloudflare(port=5000)
public_url = getattr(tunnel_info, 'tunnel', None) or getattr(tunnel_info, 'url', None) or str(tunnel_info)

print("=" * 75)
print("🚀 NetWatch Web App is LIVE on Cloudflare!")
print("=" * 75)
print(f"\n👉 Dashboard Home:    {public_url}")
print(f"👉 Ensemble Scorer:   {public_url}/ensemble")
print(f"👉 PCAP Upload:       {public_url}/upload")
print(f"👉 Test Center:       {public_url}/test_center")
print("=" * 75)